# B-spline explorer — curvature check and H-conditioning

The pipeline starts from the central sphere congruence, whose radii are
`r = 1/H`. So a reference surface is only usable if its **mean curvature never
vanishes and never changes sign** — an `H` through zero sends the radii through
infinity. This notebook is for checking that before committing to a full run,
and for trying the stage-1b optimisation that pushes `|H|` off zero.

Surfaces are addressed by **index** into `SURFACES`, so you can sweep by number
rather than typing names.

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
CHAKANA = (ROOT / "Chakana_Geo").resolve()
for sub in ["hanan/src", "kayviz/src"]:
    p = str(CHAKANA / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import matplotlib.pyplot as plt
import kayviz as kv

from moebius.pipeline import (
    MoebiusState, load_surface, sample_selection,
    setup_bspline_optimizer, run_bspline_optimizer,
)
from moebius.utils.bsplines import (
    read_bspline_json, save_bspline_to_json,
    sample_bspline_surface, bspline_curvatures,
)

BSPLINE_DIR = ROOT / "data" / "bsplines"

# Every surface in data/bsplines, sorted: index into this list instead of
# spelling out names.  SURFACES[i] is the name, without the .json.
SURFACES = sorted(p.stem for p in BSPLINE_DIR.glob("*.json"))
GENERATED = [n for n in SURFACES if n.startswith("surface_")]   # the surface_000i set
NAMED     = [n for n in SURFACES if not n.startswith("surface_")]

print(f"{len(SURFACES)} surfaces: {len(GENERATED)} generated, {len(NAMED)} named")
print(f"  generated: {GENERATED[0]} .. {GENERATED[-1]}")
print(f"  named    : {', '.join(NAMED[:8])}{' ...' if len(NAMED) > 8 else ''}")

kv.init()
kv.clear()

---
## 1 — Curvature of one surface

Pick by index. `SURFACES[i]` is the whole list; use `GENERATED[i]` to stay
within the `surface_000i` set.

In [ ]:
# ── pick a surface ────────────────────────────────────────────────────────────
INDEX    = 2                 # index into the list below
USE_LIST = GENERATED         # GENERATED, NAMED, or SURFACES
U_SAMPLE = V_SAMPLE = 40
# ─────────────────────────────────────────────────────────────────────────────

name = USE_LIST[INDEX]
bsp  = read_bspline_json(str(BSPLINE_DIR / f"{name}.json"))
u = np.linspace(0, 1, U_SAMPLE)
v = np.linspace(0, 1, V_SAMPLE)

V, F = sample_bspline_surface(bsp, u, v)
K, H, n = bspline_curvatures(bsp, u, v)

h = H.ravel()
sign_change = h.min() * h.max() < 0
print(f"[{INDEX}] {name}")
print(f"  degree {bsp.order(0)-1}x{bsp.order(1)-1}, control points {bsp.controlpoints.shape[:2]}")
print(f"  H  min {h.min():+.4f}  max {h.max():+.4f}  |H| min {np.abs(h).min():.4e}")
print(f"  K  min {K.min():+.4f}  max {K.max():+.4f}")
print(f"  => {'H CHANGES SIGN - not usable as is' if sign_change else 'H is one-signed - usable'}")
if not sign_change and np.abs(h).min() < 1e-2:
    print(f"     but |H| gets as small as {np.abs(h).min():.2e}; r = 1/H will be large there")

kv.register_surface_mesh(name, V, F)
kv.add_scalar_quantity(name, "H", h,          defined_on="vertices")
kv.add_scalar_quantity(name, "K", K.ravel(),  defined_on="vertices")
kv.add_scalar_quantity(name, "|H|", np.abs(h), defined_on="vertices")
kv.add_scalar_quantity(name, "r = 1/H", 1.0 / np.where(np.abs(h) > 1e-9, h, np.nan),
                        defined_on="vertices")
kv.add_vector_quantity(name, "normal", n.reshape(-1, 3), defined_on="vertices")
kv.show()

---
## 2 — Sweep the whole set

Which surfaces are usable at all. This is the check behind the note in the
README that `data/bsplines/` still needs reducing to surfaces with `H != 0`.

In [ ]:
# ── sweep ─────────────────────────────────────────────────────────────────────
SWEEP_LIST   = SURFACES      # GENERATED, NAMED, or SURFACES
SWEEP_SAMPLE = 25            # coarser grid; this is a screen, not a final answer
# ─────────────────────────────────────────────────────────────────────────────

uu = np.linspace(0, 1, SWEEP_SAMPLE)
rows = []
for i, nm in enumerate(SWEEP_LIST):
    try:
        b = read_bspline_json(str(BSPLINE_DIR / f"{nm}.json"))
        _, Hs, _ = bspline_curvatures(b, uu, uu)
        hs = Hs.ravel()
        rows.append((i, nm, hs.min(), hs.max(), np.abs(hs).min(),
                     bool(hs.min() * hs.max() < 0)))
    except Exception as e:
        rows.append((i, nm, np.nan, np.nan, np.nan, None))
        print(f"  [{i}] {nm}: {type(e).__name__}: {str(e)[:60]}")

ok  = [r for r in rows if r[5] is False]
bad = [r for r in rows if r[5] is True]
err = [r for r in rows if r[5] is None]
print(f"\n{len(rows)} surfaces: {len(ok)} one-signed H, {len(bad)} sign change, {len(err)} failed to read")
print("\nusable (index: name, |H| min):")
for i, nm, lo, hi, amin, _ in ok:
    print(f"  {i:3d}: {nm:22s} |H|min {amin:.3e}   H in [{lo:+.3f}, {hi:+.3f}]")

USABLE = [nm for _, nm, _, _, _, _ in ok]      # feed this back into USE_LIST

In [ ]:
# distribution of the worst |H| per surface — how close to the limit each one is
vals = np.array([r[4] for r in rows if r[5] is not None])
flag = np.array([r[5] for r in rows if r[5] is not None])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist([vals[~flag], vals[flag]], bins=30, stacked=True,
           label=["one-signed H", "H changes sign"])
ax[0].set_xlabel("min |H| over the sampled grid"); ax[0].set_ylabel("surfaces")
ax[0].legend(); ax[0].set_title("how close to H = 0")
lo = np.array([r[2] for r in rows if r[5] is not None])
hi = np.array([r[3] for r in rows if r[5] is not None])
ax[1].scatter(lo[~flag], hi[~flag], s=14, label="one-signed H")
ax[1].scatter(lo[flag],  hi[flag],  s=14, label="H changes sign")
ax[1].axhline(0, lw=.6, c="k"); ax[1].axvline(0, lw=.6, c="k")
ax[1].set_xlabel("H min"); ax[1].set_ylabel("H max"); ax[1].legend()
ax[1].set_title("sign change = straddling an axis")
plt.tight_layout(); plt.show()

---
## 3 — Push `|H|` away from zero (stage 1b)

`setup_bspline_optimizer` deforms the surface's **control points** so `|H|`
rises above `H_THRESH`, keeping `r = 1/H` well conditioned. It uses
`MeanCurvatureBspline` with step control and Laplacian smoothing on the control
grid, so the deformation stays gentle.

It obviously helps where `H` merely comes close to zero, but it can also remove
an outright **sign change**, provided the region on the wrong side is small.
`surface_00002` samples `H` over `[-6.84, +6.05]`, and this stage still drove its
12 offending samples across and left `H` entirely negative, moving the surface by
a median of 1.1e-06. What decides the outcome is how much of the surface has to
move, not whether the sign formally changes: a large region of the opposite sign
cannot be nudged out, and the run then either fails to converge or returns
something no longer resembling the input.

So a sign change in section 2 is a reason to try this stage, not to give up.
Judge it by the before/after below — `|H| min`, the sign change, and how far the
surface moved.


In [ ]:
# ── parameters ────────────────────────────────────────────────────────────────
OPT_INDEX  = INDEX           # which surface; defaults to the one above
OPT_LIST   = USE_LIST
H_THRESH   = 0.05            # target: |H| >= this
W_H        = 1.0
W_STEP     = 0.1             # cap on control-point movement per iteration
OPT_ITER   = 30
# ─────────────────────────────────────────────────────────────────────────────

opt_name = OPT_LIST[OPT_INDEX]
state = load_surface(opt_name, "bspline_opt",
                     u_sample_num=U_SAMPLE, v_sample_num=V_SAMPLE)

cp_before = state.bspline.controlpoints.copy()
V0, F0 = sample_bspline_surface(state.bspline, state.u_pts, state.v_pts)
_, H0, _ = bspline_curvatures(state.bspline, state.u_pts, state.v_pts)

setup_bspline_optimizer(state, H_thresh=H_THRESH, w_H=W_H, w_step=W_STEP)
run_bspline_optimizer(state, max_iter=OPT_ITER)

V1, F1 = sample_bspline_surface(state.bspline, state.u_pts, state.v_pts)
_, H1, _ = bspline_curvatures(state.bspline, state.u_pts, state.v_pts)
h0, h1 = H0.ravel(), H1.ravel()

print(f"\n{opt_name}")
print(f"  |H| min   {np.abs(h0).min():.4e}  ->  {np.abs(h1).min():.4e}   (target {H_THRESH})")
print(f"  H range   [{h0.min():+.4f}, {h0.max():+.4f}]  ->  [{h1.min():+.4f}, {h1.max():+.4f}]")
print(f"  below threshold: {int((np.abs(h0) < H_THRESH).sum())}  ->  {int((np.abs(h1) < H_THRESH).sum())}"
      f"  of {h0.size} samples")
print(f"  sign change: {h0.min()*h0.max() < 0}  ->  {h1.min()*h1.max() < 0}")
print(f"  surface moved: median {np.median(np.linalg.norm(V1-V0, axis=1)):.4e}"
      f"  max {np.linalg.norm(V1-V0, axis=1).max():.4e}")
print(f"  control points moved: max {np.abs(state.bspline.controlpoints - cp_before).max():.4e}")

kv.register_surface_mesh("before", V0, F0)
kv.add_scalar_quantity("before", "H", h0, defined_on="vertices")
kv.register_surface_mesh("after",  V1, F1)
kv.add_scalar_quantity("after", "H", h1, defined_on="vertices")
kv.add_scalar_quantity("after", "moved", np.linalg.norm(V1-V0, axis=1), defined_on="vertices")
kv.show()

In [ ]:
# before / after, as histograms of H
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(h0, bins=60, alpha=.6, label="before")
ax.hist(h1, bins=60, alpha=.6, label="after")
for x in (-H_THRESH, H_THRESH):
    ax.axvline(x, c="k", lw=.8, ls="--")
ax.axvline(0, c="r", lw=1)
ax.set_xlabel("mean curvature H"); ax.set_ylabel("samples")
ax.set_title(f"{opt_name}: dashed = +/- H_THRESH, red = 0")
ax.legend(); plt.tight_layout(); plt.show()

---
## 4 — Save the conditioned surface

Writes the deformed control points back out in the same JSON format, so the
result can be fed to the main pipeline notebook as an ordinary input.

In [ ]:
SAVE_AS = f"{opt_name}_Hfix"      # written to data/bsplines/<SAVE_AS>.json

out = BSPLINE_DIR / f"{SAVE_AS}.json"
save_bspline_to_json(state.bspline, str(out))
print(f"wrote {out}")

# Verify the round trip exactly: normalize=False, because the reader rescales to
# a bounding box of 2 by default and the deformation changed the bounding box, so
# a normalised reload is a *different* surface by design, not a failed write.
raw = read_bspline_json(str(out), normalize=False)
dV = np.abs(np.asarray(raw(state.u_pts, state.v_pts))
            - np.asarray(state.bspline(state.u_pts, state.v_pts))).max()
print(f"  round trip: control points {raw.controlpoints.shape[:2]}, max|dV| {dV:.3e}")
assert dV < 1e-10, "save/read round trip is lossy"

# And what the pipeline itself will see, loading it the normal way.
check = read_bspline_json(str(out))
_, Hc, _ = bspline_curvatures(check, state.u_pts, state.v_pts)
print(f"  as the pipeline loads it: |H| min {np.abs(Hc).min():.4e}, "
      f"sign change {bool(Hc.min() * Hc.max() < 0)}")
print(f"  (|H| is scaled by the normalisation, the sign change is not)")
